# Data Pre-Processing

## First dataset

The first dataset was obtained from this link in Kaggle:
https://www.kaggle.com/datasets/atulyakumar98/gundetection/data

All the labels and images were in one folder. Hence, I placed 'txt' files into one folder called **'labels'**, and all the 'jpg' or 'png' files into another folder called **'images'**. Then each folder was split into two -- 80% for a **train** set and 20% for a **test** set.

The final structure became as follows: one main **data** folder with two sub-folders; **train** and **test**; and each with two more sub-folders, namely **images** and **labels**. Unfortunately, a valid set was not created at this juncture as we failed to understand it's importance. Moving forward with the new datasets, a valid folder was created and tested accordingly.

The code that was used for the splitting is attached below.

To run the code:
Download the **"archive.zip**" file from the "unprocessed data" folder in the google drive submitted: https://drive.google.com/drive/folders/16zvEIO6jqvb1qOD5xqyHjMwvtT90V5sg?usp=drive_link


Ensure the file is downloaded onto your computer. Then, upload the downloaded data onto this runtime, and then proceed to run the code below.

In [8]:
# import libraries
import os
import random
import shutil

import cv2
import numpy as np
from tqdm import tqdm

In [12]:
# unzip data
!unzip -q archive.zip -d gun_data/

unzip:  cannot find or open archive.zip, archive.zip.zip or archive.zip.ZIP.


In [13]:
# labelling directories
DATA_DIR = "gun_data"
IMAGES_DIR = DATA_DIR
LABELS_DIR = DATA_DIR
OUTPUT_DIR = "/content/final_gun_data"
TRAIN_RATIO = 0.8  # 80% train, 20% test, no validation for now

# creating needed directories
image_files = [f for f in os.listdir(IMAGES_DIR) if f.endswith((".jpg", ".png"))]
random.shuffle(image_files)

split_index = int(len(image_files) * TRAIN_RATIO)
train_files = image_files[:split_index]
test_files = image_files[split_index:]

def create_dirs(subset):
    os.makedirs(os.path.join(OUTPUT_DIR, subset, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, subset, "labels"), exist_ok=True)

create_dirs("train")
create_dirs("test")

# copying files into the correct directories
def copy_files(file_list, subset):
    for filename in file_list:
        image_path = os.path.join(IMAGES_DIR, filename)
        label_path = os.path.join(LABELS_DIR, os.path.splitext(filename)[0] + ".txt")

        dest_image = os.path.join(OUTPUT_DIR, subset, "images", filename)
        dest_label = os.path.join(OUTPUT_DIR, subset, "labels", os.path.splitext(filename)[0] + ".txt")

        if os.path.exists(label_path):
            shutil.copyfile(image_path, dest_image)
            shutil.copyfile(label_path, dest_label)
        else:
            print(f"[SKIPPED] No label for {filename}")

copy_files(train_files, "train")
copy_files(test_files, "test")

print("Dataset split complete.")


FileNotFoundError: [Errno 2] No such file or directory: 'gun_data'

## Second Dataset

The second dataset was obtained from this link in kaggle: https://www.kaggle.com/datasets/iqmansingh/guns-knives-object-detection?select=guns-knives-yolo

In this dataset, the knife is class 0 and gun is class 1. However, in the original dataset above, the gun is class 0, and there were no knives. to standardise this, the above dataset, "gun_data", was edited to change the labels of the gun from '0' to '1'. The code is as below:

In [14]:
# function to relable class instances
def relabel_class_indices(label_dir):
    for root, _, files in os.walk(label_dir):
        for file in files:
            if file.endswith(".txt"):
                file_path = os.path.join(root, file)
                with open(file_path, "r") as f:
                    lines = f.readlines()

                new_lines = []
                for line in lines:
                    if line.strip():
                        parts = line.strip().split()
                        # Change class 0 → 1 (gun)
                        if parts[0] == '0':
                            parts[0] = '1'
                        new_lines.append(" ".join(parts) + "\n")

                with open(file_path, "w") as f:
                    f.writelines(new_lines)

# calling function for 'labels' folder in both train and test folders
relabel_class_indices("/content/final_gun_data/split/test/labels")
relabel_class_indices("/content/final_gun_data/split/train/labels")

Now that the **"final_gun_data"** folder is in the correct format, it can be combined with the second dataset. This was done by downloading both folders to a computer's local drive, and manually copy pasting all the images and labels from the train and test folders of both datasets into a new folder called **'final_data'**.

Due to the number of images and labels in each folder from each dataset, the current numbers of knives and guns in each folder was counted as follows:
  
  knife: 216
  
  gun: 924


---



Train Set:
  
  knife: 2766
  
  gun: 5084


---


Valid Set:
  
  knife: 601
  
  gun: 602

This shows a massive disbalance. Data augmentation is performed to resolve this.

## Data Augmentation

The following code augments the data by finding images that contain at least one knife and applying random augmentation, such as flipping, rotating, changing hues and brightness. It then saves the new image with a suffix at the back, and copies the original .txt files while adding the same suffix (as the bounding box should still approximately be the same).

In order to run the following code, the zip file **'final_gun_knife_data.zip'** has to be uploaded. It can be found in the "Unprocessed Data" folder, linked here: https://drive.google.com/file/d/1wAKsRayqExBdEt0HWgxYCg--b5diCTZH/view?usp=drive_link

It is then unzipped using the following code:

In [15]:
# unzip data
!unzip -q final_gun_knife_data.zip -d imbalanced_data/

unzip:  cannot find or open final_gun_knife_data.zip, final_gun_knife_data.zip.zip or final_gun_knife_data.zip.ZIP.


In [16]:
# Paths
base_dir = "/content/imbalanced_data/final_data"
sets = ['train', 'valid', 'test']
target_class = 0  # knife class ID

# Augmentation functions

# adjusting hues/colours randomly, by either making it 10% brighter or darker
def adjust_hsv(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[..., 0] = (hsv[..., 0] + random.uniform(-10, 10)) % 180
    hsv[..., 1] *= random.uniform(0.8, 1.2)
    hsv[..., 2] *= random.uniform(0.7, 1.3)
    hsv = np.clip(hsv, 0, 255).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

# rotating images
def rotate_image(img, angle):
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), borderValue=(114, 114, 114))

# flipping images
def augment_image(img):
    if random.random() < 0.5:
        img = cv2.flip(img, 1)
    img = rotate_image(img, random.uniform(-25, 25))
    img = adjust_hsv(img)
    return img

# augment to balance
for s in sets:
    print(f"\n Processing set: {s}")
    img_dir = os.path.join(base_dir, s, "images")
    lbl_dir = os.path.join(base_dir, s, "labels")

    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    images = [f for f in os.listdir(img_dir) if f.endswith('.jpg') or f.endswith('.png')]
    knife_imgs, gun_count = [], 0

    for img_file in images:
        label_path = os.path.join(lbl_dir, os.path.splitext(img_file)[0] + ".txt")
        if not os.path.exists(label_path):
            continue
        with open(label_path, "r") as f:
            labels = f.readlines()

        has_knife = any(line.startswith(str(target_class)) for line in labels)
        has_gun = any(line.startswith("1") for line in labels)

        if has_knife:
            knife_imgs.append(img_file)
        if has_gun:
            gun_count += 1

    print(f" Knife images: {len(knife_imgs)} | Gun images: {gun_count}")

    current_knife_count = len(knife_imgs)
    required_augments = gun_count - current_knife_count
    if required_augments <= 0:
        print(" Already balanced or over-sampled.")
        continue

    print(f"Augmenting {required_augments} knife images...")

    # Augment randomly selected knife images
    aug_count = 0
    i = 0
    while aug_count < required_augments:
        img_file = knife_imgs[i % current_knife_count]
        base_name = os.path.splitext(img_file)[0]
        img_path = os.path.join(img_dir, img_file)
        label_path = os.path.join(lbl_dir, base_name + ".txt")

        img = cv2.imread(img_path)
        with open(label_path, "r") as f:
            label_data = f.read()

        aug_img = augment_image(img.copy())
        aug_name = f"{base_name}_aug{aug_count}"
        aug_img_path = os.path.join(img_dir, f"{aug_name}.jpg")
        aug_lbl_path = os.path.join(lbl_dir, f"{aug_name}.txt")

        cv2.imwrite(aug_img_path, aug_img)
        with open(aug_lbl_path, "w") as f:
            f.write(label_data)

        aug_count += 1
        i += 1

    print(f"Added {aug_count} new knife images for balance.")



 Processing set: train
 Knife images: 0 | Gun images: 0
 Already balanced or over-sampled.

 Processing set: valid
 Knife images: 0 | Gun images: 0
 Already balanced or over-sampled.

 Processing set: test
 Knife images: 0 | Gun images: 0
 Already balanced or over-sampled.


In [17]:
import shutil
from google.colab import files

# Create ZIP of final_data
zip_path = "/content/final_data.zip"
shutil.make_archive("/content/final_data", 'zip', base_dir)

# Download
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>